In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import os
import json

from tqdm import tqdm

/project/GCRB/Hon_lab/s223695/anaconda3/lib/python3.10/site-packages/pandas/core/arrays/masked.py:60: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


In [2]:
json_fp = "./config.json"
with open(json_fp, 'r') as fp:
    config = json.load(fp)

anno_bed_file = annotation_file = config["input_data"]["anno_bed_file"]
annotation_file = config["user_defined_data"]["annotation_file"]

In [11]:
oligo_file_1 = "/project/shared/gcrb_igvf/shared/sgRNA_design/Cardiac.Production.2/final.2024-03-14/oligo_pool1.genes.oligos"
oligo_file_2 = "/project/shared/gcrb_igvf/shared/sgRNA_design/Cardiac.Production.2/final.2024-03-14/oligo_pool2.other.oligos"

In [5]:
if os.path.exists(annotation_file):
    annotation_df = pd.read_csv(annotation_file)
else:
    bed_df = pd.read_csv(anno_bed_file,sep="\t",names=["guide_chr","guide_start","guide_end","attr","score","strand"])
    bed_df["protospacer_target"]     = bed_df["attr"].apply(lambda x: x.split(";")[0])
    bed_df["intended_target_region"] = bed_df["attr"].apply(lambda x: x.split(";")[1])
    bed_df["gene_target"]            = bed_df["attr"].apply(lambda x: x.split(";")[2])
    bed_df["source"]                 = bed_df["attr"].apply(lambda x: x.split(";")[3])
    bed_df = bed_df.drop("attr",axis=1)
    bed_df.to_csv(annotation_file,index=None)
    annotation_df = bed_df
    
annotation_df.head()

,Unnamed: 0.1,Unnamed: 0,guide_chr,guide_start,guide_end,score,strand,protospacer_target,intended_target_region,gene_target,source,closest_gene,closest_dist,closest_gene_target,closest_dist_target
0,0,0,chr9,130713821,130713839,.,+,chr9:130713821-130713839(+),chr9:130713809-130714264,ABL1_ENST00000372348.2,"OpenTargets,CHDgene.au",ABL1,993,ABL1,993
1,1,1,chr9,130713809,130713827,.,+,chr9:130713809-130713827(+),chr9:130713809-130714264,ABL1_ENST00000372348.2,"OpenTargets,CHDgene.au",ABL1,993,ABL1,993
2,2,2,chr9,130713859,130713877,.,+,chr9:130713859-130713877(+),chr9:130713809-130714264,ABL1_ENST00000372348.2,"OpenTargets,CHDgene.au",ABL1,993,ABL1,993
3,3,3,chr9,130714246,130714264,.,-,chr9:130714246-130714264(-),chr9:130713809-130714264,ABL1_ENST00000372348.2,"OpenTargets,CHDgene.au",ABL1,993,ABL1,993
4,4,4,chr9,130713865,130713883,.,+,chr9:130713865-130713883(+),chr9:130713809-130714264,ABL1_ENST00000372348.2,"OpenTargets,CHDgene.au",ABL1,993,ABL1,993


In [33]:
oligo_df_1 = pd.read_csv(oligo_file_1,sep="\t",header=None,names=["name","oligo"])
oligo_df_2 = pd.read_csv(oligo_file_2,sep="\t",header=None,names=["name","oligo"])
oligo_df = pd.concat([oligo_df_1,oligo_df_2])

In [58]:
oligo_df_1["name"].iloc[1]

'chr9:130713809-130713827(+);chr9:130713809-130714264;ABL1_ENST00000372348.2;OpenTargets,CHDgene.au;CAAAGCAGAGAAGCGAGAG'

In [61]:
oligo_name = oligo_df["name"].apply(lambda x: np.array(x.split(";"))[[0,4,2,1]])

In [73]:
oligo_name_anno_df = pd.DataFrame(np.concatenate(oligo_name.values).reshape(-1,4),
                                  columns=["gRNA_name","oligo_seq","target_name","target_region"]
                                 )

In [76]:
oligo_name_anno_df

,gRNA_name,oligo_seq,target_name,target_region
0,chr9:130713821-130713839(+),GCGAGAGCGGCCACTAGTT,ABL1_ENST00000372348.2,chr9:130713809-130714264
1,chr9:130713809-130713827(+),CAAAGCAGAGAAGCGAGAG,ABL1_ENST00000372348.2,chr9:130713809-130714264
2,chr9:130713859-130713877(+),AGATGAAGAAGCTAAGATA,ABL1_ENST00000372348.2,chr9:130713809-130714264
3,chr9:130714246-130714264(-),CAGCGAGCGTAGAGGGAAA,ABL1_ENST00000372348.2,chr9:130713809-130714264
4,chr9:130713865-130713883(+),AGAAGCTAAGATAGGGGGT,ABL1_ENST00000372348.2,chr9:130713809-130714264
...,...,...,...,...
14325,chr9:22115301-22115320(-),TACAAAATATGCTCACTAA,Element56,chr9:22114937-22115637
14326,chr9:22115389-22115408(+),gcgatatatgatcactcCA,Element56,chr9:22114937-22115637
14327,chr9:22115445-22115464(-),ATGTACTTGGTGATGTAAA,Element56,chr9:22114937-22115637
14328,chr9:22115499-22115518(+),AAACCTGAAAAAAGCAATG,Element56,chr9:22114937-22115637


In [75]:
oligo_name_anno_df.to_csv("oligo_pool_info.csv")